# 🕵️ Dia 65 — Sinais de phishing e como inspecionar um link sem clicar

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 65* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-65.md) (*Phishing red flags and inspecting a link without clicking it*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Segunda-feira, 19 de outubro. A segunda onda chegou: um e-mail *"Cadastro Pix suspenso"* e um WhatsApp com o logotipo da Serra Azul. O link do e-mail termina em `?c=...`.

Rafa: *"Se você clicar, quem fica sabendo?"*

Neste notebook você vai:

1. **Extrair os links do HTML** sem renderizar nada e comparar texto visível com `href`.
2. **Desarmar** links, **decodificar** o token de rastreamento e **removê-lo** antes de qualquer consulta.
3. Ver como o **punycode** engana o olho.
4. Ler a **cadeia de redirecionamentos** simulada e achar o salto que escondeu o destino.
5. Jogar o **quiz dos dez sinais** e gerar a planilha `notas/dia65-planilha-links.md`.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

## 1. A amostra, preservada

O original manda salvar o `.eml` antes de tudo. Aqui a amostra 2 da segunda onda vem **embutida** no notebook (plano B offline). Se houver internet, a célula tenta baixar a mesma amostra do repositório; se falhar, usa a cópia embutida. Nos dois casos, calculamos o SHA-256 **antes** de mexer em qualquer coisa.

In [ ]:
import hashlib                      # impressão digital da evidência
import urllib.request               # só para tentar baixar a amostra (opcional)

# Cópia embutida da amostra 2 (fictícia). É o plano B se não houver rede.
AMOSTRA_EMBUTIDA = """From: "Serra Azul Seguranca" <aviso@serraazul-seguranca.example>
Reply-To: atendimento.serraazul@webmail.example
Return-Path: <bounce-77@mailer-rapido.example>
To: cooperado17452@correio.example
Subject: Cadastro Pix suspenso: atualize em ate 2 horas
Date: Mon, 19 Oct 2026 07:41:09 -0300
MIME-Version: 1.0
Content-Type: text/html; charset="utf-8"

<html><body>
<p>Prezado(a) cooperado(a),</p>
<p>Identificamos uma tentativa de acesso ao seu cadastro Pix a partir de um novo aparelho.
Para evitar o bloqueio definitivo da sua chave, atualize seus dados em
<a href="https://lnk.example/sa9?c=Y29vcC0xNzQ1Mg==">https://www.serraazul.example/pix/atualizar</a>
ate as 10h de hoje.</p>
<p>Comprovante do bloqueio em anexo (comprovante-bloqueio.html).</p>
<p>Equipe de Seguranca</p>
</body></html>
"""

URL_RAW = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
           "main/caso-continuo/dados/dia-65/amostra-2-email.eml")

try:
    # timeout curto: se a rede estiver lenta ou bloqueada, seguimos offline
    with urllib.request.urlopen(URL_RAW, timeout=5) as resposta:
        amostra = resposta.read().decode("utf-8")
    origem = "repositório (GitHub raw)"
except Exception as erro:          # sem rede, 404, proxy... tanto faz: plano B
    amostra = AMOSTRA_EMBUTIDA
    origem = f"cópia embutida (rede indisponível: {type(erro).__name__})"

# O hash vem ANTES da análise: é o que prova que você analisou o que recebeu.
hash_amostra = hashlib.sha256(amostra.encode("utf-8")).hexdigest()
print("Origem da amostra:", origem)
print("SHA-256:", hash_amostra)

## 2. Separar cabeçalho e corpo com o módulo `email`

O Python lê `.eml` nativamente. Vamos olhar os três endereços que o checklist compara: **From**, **Reply-To** e **Return-Path**.

In [ ]:
from email import message_from_string     # parser de e-mail da biblioteca padrão
from email.utils import parseaddr          # separa "Nome" <endereco>

msg = message_from_string(amostra)         # vira um objeto com cabeçalhos e corpo

nome_exibicao, endereco_from = parseaddr(msg["From"])   # o nome que a vítima vê × o endereço real
_, endereco_reply = parseaddr(msg["Reply-To"] or "")     # para onde vai a resposta
_, endereco_return = parseaddr(msg["Return-Path"] or "") # para onde voltam os erros de entrega

def dominio(endereco):
    """Devolve só o que vem depois do @, em minúsculas."""
    return endereco.rsplit("@", 1)[-1].lower() if "@" in endereco else ""

print(f"Nome de exibição : {nome_exibicao}")
print(f"From             : {endereco_from}  (domínio {dominio(endereco_from)})")
print(f"Reply-To         : {endereco_reply}  (domínio {dominio(endereco_reply)})")
print(f"Return-Path      : {endereco_return}  (domínio {dominio(endereco_return)})")
print(f"Assunto          : {msg['Subject']}")

# Sinal 9 do checklist: Reply-To diferente do From.
if endereco_reply and dominio(endereco_reply) != dominio(endereco_from):
    print("\n⚠️  Sinal 9: Reply-To aponta para outro domínio.")

## 3. Texto do link × `href`, sem abrir nada

`html.parser` lê a marcação como texto. Ele **não executa** JavaScript, **não baixa** imagens e **não segue** links. É o equivalente em código ao "exibir código-fonte" do original.

In [ ]:
from html.parser import HTMLParser

class ColetorDeLinks(HTMLParser):
    """Guarda pares (texto visível, href real) de cada <a>."""
    def __init__(self):
        super().__init__()
        self.links = []          # lista de (texto, href)
        self._href_aberto = None # href do <a> que está aberto agora
        self._texto = []         # pedaços de texto dentro desse <a>

    def handle_starttag(self, tag, attrs):
        if tag == "a":                                  # só nos interessa o <a>
            self._href_aberto = dict(attrs).get("href", "")
            self._texto = []

    def handle_data(self, data):
        if self._href_aberto is not None:               # estamos dentro de um <a>
            self._texto.append(data)

    def handle_endtag(self, tag):
        if tag == "a" and self._href_aberto is not None:
            self.links.append(("".join(self._texto).strip(), self._href_aberto))
            self._href_aberto = None

corpo = msg.get_payload()          # o HTML do e-mail (texto puro, não renderizado)
coletor = ColetorDeLinks()
coletor.feed(corpo)

from urllib.parse import urlsplit

for texto, href in coletor.links:
    host_texto = urlsplit(texto).hostname if texto.startswith("http") else None
    host_href = urlsplit(href).hostname
    print("Texto visível:", texto)
    print("href real    :", href)
    # Sinal 2: o domínio que a vítima lê não é o domínio para onde ela vai.
    print("Divergência  :", "SIM ⚠️" if host_texto != host_href else "não", f"({host_texto} × {host_href})")

## 4. Desarmar, decodificar o token e removê-lo

Três funções pequenas, cada uma com uma tarefa:

- `desarmar()` troca `http` por `hxxp` e `.` por `[.]`, para que ninguém clique por acidente;
- `ler_token()` decodifica o parâmetro em base64 (é assim que o `staff12` do original foi descoberto);
- `sem_token()` remonta a URL **sem** query string nem fragmento: é essa a versão que pode ir para uma *sandbox* privada.

In [ ]:
import base64
from urllib.parse import urlsplit, urlunsplit, parse_qs

def desarmar(url: str) -> str:
    """Versão não clicável da URL, no padrão usado em relatórios de CTI."""
    return url.replace("http", "hxxp", 1).replace(".", "[.]")

def ler_token(url: str) -> dict:
    """Tenta decodificar cada parâmetro da query como base64."""
    resultado = {}
    for chave, valores in parse_qs(urlsplit(url).query).items():
        for valor in valores:
            try:
                # validate=True: só aceita se for base64 de verdade
                resultado[chave] = base64.b64decode(valor, validate=True).decode("utf-8")
            except Exception:
                resultado[chave] = f"(não é base64) {valor}"
    return resultado

def sem_token(url: str) -> str:
    """Remove query (?...) e fragmento (#...): sobra só esquema, host e caminho."""
    p = urlsplit(url)
    return urlunsplit((p.scheme, p.netloc, p.path, "", ""))

for _, href in coletor.links:
    print("Desarmado      :", desarmar(href))
    print("Token decodif. :", ler_token(href))
    print("Para a sandbox :", desarmar(sem_token(href)))

> 💡 O token decodificado é `coop-17452`: o número do cooperado que recebeu o e-mail. Se esse link fosse clicado, ou colado num verificador **público**, o "Pescador" saberia que o cooperado 17452 (ou alguém investigando por ele) abriu a mensagem. É o `staff12` do original, versão Serra Azul.

## 5. Punycode: quando o olho mente

O sinal 3 do checklist fala de domínios em `xn--`. O navegador às vezes mostra a forma Unicode, e um `а` cirílico é idêntico a um `a` latino na tela. O codec `idna` do Python mostra as duas formas.

In [ ]:
# Dois domínios que parecem iguais para um humano
legitimo = "serraazul.example"
imitacao = "serrаazul.example"    # а é o "a" CIRÍLICO, não o latino

print("Na tela:", legitimo, "|", imitacao)
print("Iguais para o Python?", legitimo == imitacao)

# encode("idna") devolve a forma ASCII que de fato vai para o DNS
print("Forma DNS do legítimo:", legitimo.encode("idna").decode())
print("Forma DNS da imitação:", imitacao.encode("idna").decode())

# Caminho inverso: um xn-- recebido num log vira a forma que a vítima viu
print("xn--serrazul-46g.example →", b"xn--serrazul-46g.example".decode("idna"))

# Regra simples de triagem: qualquer caractere fora do ASCII num domínio merece olhar humano
for d in (legitimo, imitacao):
    suspeito = any(ord(ch) > 127 for ch in d)
    print(f"{d!r}: {'⚠️ tem caractere não ASCII' if suspeito else 'só ASCII'}")

## 6. A cadeia de redirecionamentos

Não existe urlscan.io para `.example`. O arquivo `cadeia-redirecionamento.csv` simula o que uma varredura **privada** do link limpo devolveria. Leia salto por salto e responda: **qual deles escondeu o destino real?**

In [ ]:
import csv, io

# Cópia embutida do CSV do caso (fictício); o arquivo também está em caso-continuo/dados/dia-65/
CADEIA_CSV = """salto,url_defangada,status_http,ip,observacao
1,hxxps://lnk[.]example/sa9,301,198.51.100.14,encurtador; token ?c= removido antes da varredura
2,hxxps://r[.]trk-cdn[.]example/go?d=sa9,302,198.51.100.90,redirecionador intermediário; é o salto que esconde o destino
3,hxxps://serraazul-seguranca[.]example/pix/,200,203.0.113.77,formulário falso de login; mesmo IP do serraazul-pix.example
"""

saltos = list(csv.DictReader(io.StringIO(CADEIA_CSV)))
for s in saltos:
    # 3xx = redirecionamento; 200 = chegou numa página
    tipo = "↪ redireciona" if s["status_http"].startswith("3") else "■ destino"
    print(f"Salto {s['salto']}: {tipo:14} {s['url_defangada']:45} IP {s['ip']}")

IP_DO_DIA_1 = "203.0.113.77"        # fato fixo do caso: IP do serraazul-pix.example
final = saltos[-1]
if final["ip"] == IP_DO_DIA_1:
    print("\n🔗 O destino final está no MESMO IP do site falso do Dia 1. Anote como aresta para o Dia 68.")

## 7. 🎮 Quiz dos dez sinais (amostra 2)

Para cada sinal do checklist do original, diga se ele está **presente** na amostra 2: **S** (presente), **N** (ausente). O notebook mostra o gabarito e **a linha que prova**. Lembre a regra do dia: sinal presente sem linha citada não vale.

Quer só ver o gabarito? `MODO_JOGO = False`.

In [ ]:
MODO_JOGO = True   # False = mostra o gabarito sem perguntar

# (sinal, presente?, evidência citada da amostra 2)
SINAIS = [
    ("1. Nome de exibição × domínio real", True,
     '"Serra Azul Seguranca" enviado de serraazul-seguranca.example (o real é serraazul.example)'),
    ("2. Texto do link × href", True,
     "texto www.serraazul.example/pix/atualizar, href lnk.example/sa9"),
    ("3. Domínio imitação", True,
     "serraazul-seguranca.example: palavra a mais no domínio da cooperativa"),
    ("4. Encurtador ou redirecionamento aberto", True,
     "lnk.example/sa9 → r.trk-cdn.example (2 saltos antes do destino)"),
    ("5. Prazo ou ameaça", True,
     '"atualize em ate 2 horas", "ate as 10h de hoje", "bloqueio definitivo"'),
    ("6. Credenciais/pagamento por canal errado", True,
     '"atualize seus dados" por link de e-mail; o destino é um formulário de login'),
    ("7. Saudação genérica", True,
     '"Prezado(a) cooperado(a)" — a cooperativa sabe o nome do cooperado'),
    ("8. Anexo inesperado", True,
     '"Comprovante do bloqueio em anexo (comprovante-bloqueio.html)": anexo HTML'),
    ("9. Reply-To diferente do From", True,
     "Reply-To atendimento.serraazul@webmail.example"),
    ("10. Transação/conta sem registro", False,
     "A mensagem cita o cadastro Pix, que o cooperado TEM. O sinal 10 é sobre algo que você não reconhece; aqui não se aplica."),
]

acertos = 0
for nome, presente, evidencia in SINAIS:
    print(f"\n{nome}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Presente? (S/N): ").strip().upper()
        ok = (r == "S") == presente
        acertos += ok
        print("   ✅ Isso." if ok else "   ❌ Olhe de novo.")
    print(f"   Gabarito: {'PRESENTE' if presente else 'AUSENTE'} — {evidencia}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(SINAIS)}  (Checkpoint BR: pelo menos 8)")

## 8. 📝 A planilha do dia

A célula gera `notas/dia65-planilha-links.md` com tudo o que você levantou: hash, endereços, link desarmado, token, cadeia, sinais com evidência e a frase de veredito. Ajuste o veredito no formulário.

In [ ]:
#@title Veredito da amostra 2 { display-mode: "form" }
veredito = "Phishing"  #@param ["Phishing", "Provável phishing", "Evidência insuficiente"]
porque = "9 dos 10 sinais presentes com linha citada; o destino final é um formulário de login no mesmo IP do site falso do Dia 1."  #@param {type:"string"}

from pathlib import Path
from datetime import datetime, timezone

linhas = [
    "# Planilha de inspeção de links — Dia 65 (Operação Isca, fictício)",
    "",
    f"Gerada em (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}",
    "",
    "## Amostra 2 — e-mail \"Cadastro Pix suspenso\"",
    f"- SHA-256 da amostra: `{hash_amostra}`",
    f"- From: `{desarmar(endereco_from)}` · Reply-To: `{desarmar(endereco_reply)}` · Return-Path: `{desarmar(endereco_return)}`",
]
for texto, href in coletor.links:
    linhas += [f"- Texto do link: `{desarmar(texto)}`",
               f"- href: `{desarmar(href)}`",
               f"- Token: `{ler_token(href)}` (removido antes de qualquer consulta)"]
linhas += ["", "| Salto | URL (desarmada) | HTTP | IP |", "|---|---|---|---|"]
linhas += [f"| {s['salto']} | `{s['url_defangada']}` | {s['status_http']} | {s['ip']} |" for s in saltos]
linhas += ["", "| Sinal | Achado | Evidência |", "|---|---|---|"]
# a evidência cita domínios: desarmamos também, para o markdown não criar link automático
linhas += [f"| {n} | {'presente' if p else 'ausente'} | {e.replace('.example', '[.]example')} |" for n, p, e in SINAIS]
linhas += ["", f"**Veredito:** {veredito}, porque {porque}", ""]

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia65-planilha-links.md")
saida.write_text("\n".join(linhas), encoding="utf-8")
print("\n".join(linhas))
print(f"\n💾 Salvo em {saida}")

# Checkpoint automático do original: nenhum link clicável na planilha
import re
clicaveis = re.findall(r"https?://", saida.read_text(encoding="utf-8"))
print("🔍 Links clicáveis na planilha:", len(clicaveis), "(tem que ser 0)")

try:
    from google.colab import files     # só existe no Colab
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### 🧠 Camada 8 — leia antes de fechar

Conte quantas vezes a amostra 2 fala de tempo: "2 horas", "10h de hoje", "definitivo". Escassez (Cialdini) mais autoridade (o nome "Segurança") empurram para a decisão rápida. A sua versão desse atalho é o "só vou ver para onde vai". O procedimento que você acabou de rodar existe para os dias com pressa.

## ✅ Antes de fechar o Dia 65

- [ ] Quiz com pelo menos 8/10
- [ ] `notas/dia65-planilha-links.md` gerada, com **0** links clicáveis
- [ ] Você sabe dizer qual salto escondeu o destino (o 2) e por quê
- [ ] As amostras 1 e 3 pontuadas à mão no README (com "não se aplica" onde couber)

**Amanhã (Dia 66):** o financeiro da cooperativa recebe um "boleto atualizado" da fornecedora. Remetente falsificado, domínio imitação ou caixa comprometida? O cabeçalho responde. E, logo depois, os laranjas.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>